# Lab 5 — Zadanie samodzielne: Analiza Przestępczości w Chicago

**Zbiór danych:** `chicago_crimes_sample.csv` (~50 000 ostatnich zdarzeń)

## Plan rozwiązania
| Krok | Temat |
|------|-------|
| 0 | Setup — SparkSession |
| 1 | Wczytanie i czyszczenie danych |
| 2 | UDF — klasyfikacja pory dnia |
| 3 | Cache, Broadcast Join, zapis do Parquet |
| 4 | Analiza statystyczna + `explain()` |
| 5 | *(Opcjonalnie)* MLlib — klasyfikacja wieloklasowa |

---
## Krok 0 — Setup SparkSession

**Co to robi:** Tworzy jedyny punkt wejścia do Sparka.  
**Dlaczego `PYSPARK_PYTHON`:** Workery muszą używać tego samego Pythona co kernel — bez tego UDF-y rzucają błędami.  
**Dlaczego `shuffle.partitions=4`:** Domyślnie 200 partycji na groupBy — na laptopie z małym plikiem to overkill.

In [ ]:
import os
import sys

# Workery muszą używać tego samego Pythona co kernel Jupytera
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Ustaw JAVA_HOME jeśli nie jest ustawiony (dopasuj do swojej instalacji)
if "JAVA_HOME" not in os.environ:
    candidates = [
        "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",  # macOS Apple Silicon
        "/usr/local/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",     # macOS Intel
        "/usr/lib/jvm/java-17-openjdk-amd64",                              # Linux Ubuntu
    ]
    for path in candidates:
        if os.path.exists(path):
            os.environ["JAVA_HOME"] = path
            print(f"JAVA_HOME ustawiony na: {path}")
            break
    else:
        print("UWAGA: JAVA_HOME nie znaleziony — ustaw ręcznie!")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

spark = SparkSession.builder \
    .appName("Chicago Crimes Analysis") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} gotowy!")

---
## Krok 1 — Wczytanie i czyszczenie danych

### 1a. Wczytanie CSV

**Dlaczego `inferSchema=True`:** Kolumny jak `Year` czy `Arrest` mają czyste wartości — Spark potrafi je poprawnie odgadnąć.  
**Pamiętaj:** `inferSchema` kosztuje podwójne skanowanie pliku. Na dużych danych lepiej podać schemat ręcznie.

In [ ]:
df_crimes = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("chicago_crimes_sample.csv")

print("=== SCHEMAT ===")
df_crimes.printSchema()

print(f"\nLiczba wierszy (surowe dane): {df_crimes.count():,}")
print("\n=== PIERWSZE 5 WIERSZY ===")
df_crimes.show(5, truncate=False)

### 1b. Usunięcie duplikatów

**Jak działa `dropDuplicates()`:** Spark robi shuffle — porównuje wiersze z różnych partycji. Stąd może być wolna na dużych danych, ale tutaj (~50k rekordów) to chwila.

In [ ]:
przed = df_crimes.count()
df_crimes = df_crimes.dropDuplicates()
po = df_crimes.count()

print(f"Przed usunięciem duplikatów: {przed:,}")
print(f"Po usunięciu duplikatów:     {po:,}")
print(f"Usunięto duplikatów:         {przed - po:,}")

### 1c. Usunięcie null-i w kluczowych kolumnach

**Dlaczego `subset=`:** Nie wyrzucamy wiersza z powodu brakującej wartości w nieistotnej kolumnie (np. `Ward`). Sprawdzamy tylko te, które są nam potrzebne do analizy.

In [ ]:
# Sprawdź, ile brakuje wartości w każdej kolumnie
print("=== BRAKUJĄCE WARTOŚCI PER KOLUMNA ===")
df_crimes.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_crimes.columns
]).show(truncate=False)

# Usuwamy null-e tylko w kluczowych kolumnach
# UWAGA: nazwy kolumn w tym CSV są w snake_case (patrz printSchema powyżej)
kluczowe_kolumny = ["id", "date", "primary_type", "location_description"]

przed = df_crimes.count()
df_crimes = df_crimes.dropna(subset=kluczowe_kolumny)
po = df_crimes.count()

print(f"\nPrzed usunięciem null-i: {przed:,}")
print(f"Po usunięciu null-i:     {po:,}")
print(f"Usunięto wierszy:        {przed - po:,}")

### 1d. Parsowanie i filtrowanie dat

**Format `MM/dd/yyyy hh:mm:ss a`:** Wynika z formatu kolumny `Date` w datasecie Chicago (np. `01/15/2023 03:45:00 PM`).  
**`a`** to symbol AM/PM w formacie Java/Spark.  
**Dlaczego `to_timestamp`, nie `to_date`:** Za chwilę potrzebujemy godziny (`F.hour()`) do klasyfikacji pory dnia — sam `date` by nie wystarczył.  
**Dlaczego nie ma wyjątku przy złym formacie:** `to_timestamp` zwraca `null` zamiast rzucać błąd — to celowe. Pozwala nam łatwo odfiltrować błędne rekordy.

In [ ]:
# Parsowanie daty -- format wynika z wartości w kolumnie `date`
# Przykład: "01/15/2023 03:45:00 PM"
df_crimes = df_crimes.withColumn(
    "Date_parsed",
    F.to_timestamp(F.col("date"), "MM/dd/yyyy hh:mm:ss a")
)

# Sprawdź ile dat się nie sparsowało (null = błędny format)
bledne_daty = df_crimes.filter(F.col("Date_parsed").isNull()).count()
print(f"Rekordów z błędną datą: {bledne_daty:,}")

# Odfiltruj błędne daty
df_crimes = df_crimes.filter(F.col("Date_parsed").isNotNull())

# Wyciągnij osobne kolumny: godzina i rok -- będą potrzebne w kolejnych krokach
df_crimes = df_crimes \
    .withColumn("Hour", F.hour("Date_parsed")) \
    .withColumn("Year", F.year("Date_parsed")) \
    .withColumn("Month", F.month("Date_parsed"))

print(f"Czyste dane: {df_crimes.count():,} rekordów")
df_crimes.select("date", "Date_parsed", "Hour", "Year", "Month").show(5)

---
## Krok 2 — UDF: klasyfikacja pory dnia

**Co to UDF (User Defined Function):** Własna funkcja Pythona, którą Spark będzie wywoływać wiersz po wierszu.  
**`@udf(StringType())`:** Dekorator rejestruje funkcję w Sparku i deklaruje, że zwraca `StringType`. Spark musi to znać z góry — nie może zgadnąć.  
**Ostrzeżenie:** UDF blokuje optymalizację Catalyst (planner Sparka) i jest wolniejszy niż wbudowane funkcje. Gdzie to możliwe, lepiej użyć `F.when().when()...`. Tutaj wymaganie zadania nakazuje UDF.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Definicja UDF -- zwykła funkcja Pythona z dekoratorem
@udf(StringType())
def pora_dnia(timestamp):
    """
    Klasyfikuje godzinę zdarzenia jako porę dnia.
    Argument `timestamp` to obiekt datetime (wynik to_timestamp).
    """
    if timestamp is None:
        return None
    hour = timestamp.hour  # wyciągamy godzinę z obiektu datetime
    if 6 <= hour < 12:
        return "ranek"
    elif 12 <= hour < 18:
        return "dzień"
    elif 18 <= hour < 22:
        return "wieczór"
    else:                  # 22-5
        return "noc"

# Zastosowanie UDF -- tworzy nową kolumnę
df_crimes = df_crimes.withColumn("time_of_day", pora_dnia(F.col("Date_parsed")))

# Weryfikacja rozkładu
print("=== ROZKŁAD PRZESTĘPSTW WG PORY DNIA ===")
df_crimes.groupBy("time_of_day") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show()

In [ ]:
# BONUS: Równoważne rozwiązanie bez UDF (szybsze w produkcji)
# Catalyst potrafi to zoptymalizować -- UDF nie.
# Pokazujemy dla porównania -- nie nadpisujemy kolumny.
df_crimes.withColumn(
    "time_of_day_no_udf",
    F.when((F.col("Hour") >= 6)  & (F.col("Hour") < 12), "ranek")
     .when((F.col("Hour") >= 12) & (F.col("Hour") < 18), "dzień")
     .when((F.col("Hour") >= 18) & (F.col("Hour") < 22), "wieczór")
     .otherwise("noc")
).select("time_of_day", "time_of_day_no_udf", "Hour").show(10)

---
## Krok 3 — Optymalizacja: Cache, Broadcast Join, Parquet

### 3a. Cache

**Dlaczego tutaj:** `df_crimes` to wynik całego potoku czyszczenia (dropDuplicates → dropna → parsowanie → UDF). Bez cache Spark powtórzyłby cały ten potok przy każdej kolejnej akcji (`groupBy`, `count`, `write`...).  
**Dlaczego `count()` po `cache()`:** `cache()` jest leniwe — samo nic nie robi. Pierwsza akcja materializuje dane w pamięci RAM executorów.

In [ ]:
import time

# Bez cache -- każda operacja przelicza od nowa
t0 = time.time()
_ = df_crimes.count()
_ = df_crimes.filter(F.col("arrest") == True).count()
t_bez = time.time() - t0
print(f"Czas BEZ cache (2 akcje): {t_bez:.2f}s")

# Cache -- dane zapisywane w RAM po pierwszym przeliczeniu
df_crimes = df_crimes.cache()
df_crimes.count()  # materializacja cache

t0 = time.time()
_ = df_crimes.count()
_ = df_crimes.filter(F.col("arrest") == True).count()
t_z = time.time() - t0
print(f"Czas Z cache  (2 akcje): {t_z:.2f}s")
print(f"Przyspieszenie: {t_bez/t_z:.1f}x")

### 3b. Broadcast Join

**Problem ze zwykłym join:** Spark musi zrobić shuffle — przetasować dane obu tabel między executorami, żeby pasujące klucze trafiły razem. To najdroższa operacja w Sparku (sieć + dysk).  
**Rozwiązanie `broadcast()`:** Mała tabela słownikowa (kategorie) jest kopiowana do KAŻDEGO executora. Każdy może joinować lokalnie, bez żadnego shuffle.

In [ ]:
from pyspark.sql.functions import broadcast

# Mała tabela słownikowa: typ przestępstwa → kategoria ogólna
# W praktyce mogłaby to być tabela z bazy danych lub mały CSV
kategorie = spark.createDataFrame([
    ("THEFT",               "Majątkowe"),
    ("BATTERY",             "Przemoc"),
    ("CRIMINAL DAMAGE",     "Majątkowe"),
    ("NARCOTICS",           "Narkotyki"),
    ("ASSAULT",             "Przemoc"),
    ("OTHER OFFENSE",       "Inne"),
    ("BURGLARY",            "Majątkowe"),
    ("MOTOR VEHICLE THEFT", "Majątkowe"),
    ("ROBBERY",             "Przemoc"),
    ("DECEPTIVE PRACTICE",  "Majątkowe"),
], ["primary_type", "Kategoria"])

# broadcast() -> Spark wysyła `kategorie` do każdego executora, zero shuffle
df_z_kategorią = df_crimes.join(
    broadcast(kategorie),
    on="primary_type",
    how="left"   # left: zachowaj wszystkie przestępstwa, nawet bez kategorii
)

# Weryfikacja -- null w Kategoria = typ nieznany w słowniku
print("=== ROZKŁAD WG KATEGORII ===")
df_z_kategorią.groupBy("Kategoria") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show()

# Sprawdź w planie -- szukaj "BroadcastHashJoin" zamiast "SortMergeJoin"
print("=== PLAN ZAPYTANIA (szukaj BroadcastHashJoin) ===")
df_z_kategorią.explain()

### 3c. Zapis do partycjonowanego Parquet

**Dlaczego Parquet, nie CSV:**
- Format kolumnowy → czytasz tylko potrzebne kolumny, nie cały wiersz
- Wbudowana kompresja → ~10x mniejszy od CSV
- Typy danych zakodowane w pliku → nie trzeba `inferSchema`

**Dlaczego `partitionBy("Year")`:** Tworzy podkatalogi `Year=2021/`, `Year=2022/`... Gdy filtrujemy po roku, Spark czyta tylko odpowiedni katalog — to *predicate pushdown*.

In [ ]:
import shutil

output_dir = "chicago_crimes_parquet"

# Usuń stary zapis jeśli istnieje
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

# Zapis -- partycjonowanie po roku
t0 = time.time()
df_crimes.write \
    .mode("overwrite") \
    .partitionBy("Year") \
    .parquet(output_dir)
print(f"Zapisano Parquet w {time.time()-t0:.2f}s")

# Struktura katalogów
print("\n=== STRUKTURA PARQUET ===")
for entry in sorted(os.listdir(output_dir)):
    print(f"  {output_dir}/{entry}/")

# Wczytaj z powrotem -- Spark sam wykryje partycje
df_parquet = spark.read.parquet(output_dir)

# Filtrowanie po roku -- predicate pushdown: czyta TYLKO katalog Year=2023
t0 = time.time()
count_2023 = df_parquet.filter(F.col("Year") == 2023).count()
print(f"\nFilter Year=2023: {count_2023:,} rekordów (czas: {time.time()-t0:.2f}s)")

---
## Krok 4 — Analiza statystyczna i `explain()`

### 4a. Najpopularniejsze typy przestępstw

In [ ]:
print("=== TOP 15: TYPY PRZESTĘPSTW ===")
df_crimes.groupBy("primary_type") \
    .agg(
        F.count("*").alias("liczba"),
        F.round(
            F.sum(F.col("arrest").cast("int")) / F.count("*") * 100, 1
        ).alias("pct_areszt")
    ) \
    .orderBy(F.col("liczba").desc()) \
    .show(15, truncate=False)

### 4b. Typy przestępstw według lokalizacji i pory dnia

**To jest "najcięższa" agregacja** — trzy kolumny w `groupBy`, dwie agregacje. Na tym uruchomimy `explain()`.

In [ ]:
print("=== PRZESTĘPSTWA WG LOKALIZACJI I PORY DNIA ===")
analiza_glowna = df_crimes \
    .groupBy("location_description", "primary_type", "time_of_day") \
    .agg(
        F.count("*").alias("liczba"),
        F.sum(F.col("arrest").cast("int")).alias("areszty")
    ) \
    .withColumn(
        "pct_areszt",
        F.round(F.col("areszty") / F.col("liczba") * 100, 1)
    ) \
    .orderBy(F.col("liczba").desc())

analiza_glowna.show(20, truncate=False)

In [ ]:
# explain() -- plan wykonania tej agregacji
# Co czytać:
#   Exchange        = shuffle (drogie!) -- musi przetasować dane wg 3 kluczy groupBy
#   HashAggregate   = fizyczna agregacja
#   *               = whole-stage code generation (zoptymalizowane przez JVM)
#   Filter nisko    = predicate pushdown (Catalyst przesunął filtr jak najwcześniej)

print("=== PLAN ZAPYTANIA DLA GŁÓWNEJ AGREGACJI ===")
analiza_glowna.explain(mode="formatted")

### 4c. Analiza trendów czasowych — Window Functions

**Window functions:** Agregacje bez zmiany liczby wierszy. Dodają informację "obok" każdego wiersza zamiast zwijać dane.

In [ ]:
# Liczba przestępstw per rok i miesiąc
miesiecznie = df_crimes \
    .groupBy("Year", "Month") \
    .agg(F.count("*").alias("liczba")) \
    .orderBy("Year", "Month")

# Window: skumulowana suma w ramach każdego roku
okno_roku = Window.partitionBy("Year").orderBy("Month") \
    .rowsBetween(Window.unboundedPreceding, 0)

miesiecznie = miesiecznie.withColumn(
    "skumulowane",
    F.sum("liczba").over(okno_roku)
)

# Ranking miesiąca w swoim roku (1 = największa liczba przestępstw)
okno_rank = Window.partitionBy("Year").orderBy(F.col("liczba").desc())
miesiecznie = miesiecznie.withColumn(
    "ranking_w_roku",
    F.row_number().over(okno_rank)
)

print("=== TRENDY MIESIĘCZNE ===")
miesiecznie.show(24)

# Który miesiąc był rekordowy w każdym roku?
print("=== REKORDOWY MIESIĄC W KAŻDYM ROKU ===")
miesiecznie.filter(F.col("ranking_w_roku") == 1) \
    .select("Year", "Month", "liczba") \
    .orderBy("Year") \
    .show()

### 4d. Analiza przez Spark SQL

To samo co `groupBy().agg()`, ale zapisane jako SQL. Oba podejścia produkują identyczny plan wykonania — Catalyst je unifikuje.

In [ ]:
# Rejestracja widoku SQL
df_crimes.createOrReplaceTempView("crimes")

# Zapytanie SQL: top lokalizacje z podziałem na porę dnia
spark.sql("""
    SELECT
        location_description,
        time_of_day,
        COUNT(*) AS liczba_przestepstw,
        ROUND(SUM(CAST(arrest AS INT)) / COUNT(*) * 100, 1) AS pct_areszt
    FROM crimes
    WHERE location_description IS NOT NULL
    GROUP BY location_description, time_of_day
    HAVING COUNT(*) > 100
    ORDER BY liczba_przestepstw DESC
    LIMIT 20
""").show(truncate=False)

---
## Krok 5 (Opcjonalnie) — MLlib: klasyfikacja rodzaju przestępstwa

**Cel:** Przewidzieć `Primary Type` na podstawie: lokalizacji, pory dnia, roku, godziny i flagi `Domestic`.  

**Dlaczego `Pipeline`:** Scala wszystkie kroki (indeksowanie → wektoryzacja → model) w jeden obiekt. `fit` tylko na treningu, `transform` na teście — bez ryzyka wycieku danych.

**Dlaczego `StringIndexer`:** MLlib operuje na liczbach. `StringIndexer` zamienia `"STREET"` → `0.0`, `"RESIDENCE"` → `1.0` itp., ucząc się mapowania z danych treningowych.

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Przygotowanie danych -- tylko potrzebne kolumny, usuń null-e
ml_cols = ["location_description", "time_of_day", "Year", "Hour",
           "domestic", "primary_type"]

df_ml = df_crimes.select(ml_cols).dropna()
print(f"Rekordów do ML: {df_ml.count():,}")
df_ml.show(5)

# Zamiana Boolean -> Int dla kolumny domestic
df_ml = df_ml.withColumn("Domestic_int", F.col("domestic").cast("int"))

In [ ]:
# Kodowanie zmiennych kategorycznych jako liczby
indexer_location = StringIndexer(
    inputCol="location_description",
    outputCol="location_idx_raw",
    handleInvalid="keep"
)
indexer_tod = StringIndexer(
    inputCol="time_of_day",
    outputCol="tod_idx_raw",
    handleInvalid="keep"
)
indexer_label = StringIndexer(
    inputCol="primary_type",
    outputCol="label",
    handleInvalid="keep"
)

# KLUCZOWE: rzutujemy wyniki StringIndexer na Double.
# StringIndexer dodaje metadane 'categorical', przez co RandomForest
# wymaga maxBins >= liczbie unikalnych wartości (109 dla location).
# Rzutowanie na Double usuwa te metadane -- Spark traktuje cechy
# jako numeryczne ciągłe, co omija ograniczenie maxBins całkowicie.
from pyspark.ml.feature import SQLTransformer
cast_transformer = SQLTransformer(
    statement="SELECT *, CAST(location_idx_raw AS DOUBLE) AS location_idx, "
              "CAST(tod_idx_raw AS DOUBLE) AS tod_idx FROM __THIS__"
)

# Złączenie wszystkich cech w jeden wektor -- wymaganie MLlib
assembler = VectorAssembler(
    inputCols=["location_idx", "tod_idx", "Year", "Hour", "Domestic_int"],
    outputCol="features"
)

# Model: Random Forest dla klasyfikacji wieloklasowej
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=30,
    maxDepth=10,
    seed=42
)

# Pipeline = sekwencja kroków, wykonywana w kolejności
pipeline = Pipeline(stages=[
    indexer_location,
    indexer_tod,
    indexer_label,
    cast_transformer,   # usuwa metadane categorical -> brak problemu z maxBins
    assembler,
    rf
])

print("Pipeline zdefiniowany — gotowy do trenowania")

In [ ]:
# Podział 80% trening / 20% test -- seed=42 dla powtarzalności
train, test = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Zbiór treningowy: {train.count():,} rekordów")
print(f"Zbiór testowy:    {test.count():,} rekordów")

# Trening -- fit tylko na danych treningowych
print("\nTrenowanie modelu...")
t0 = time.time()
model = pipeline.fit(train)
print(f"Czas trenowania: {time.time()-t0:.1f}s")

In [ ]:
# Predykcja na zbiorze testowym
predictions = model.transform(test)

# Ewaluacja -- accuracy (procent poprawnych klasyfikacji)
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)
print(f"Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")

# Ważność cech -- które atrybuty najbardziej pomagają w klasyfikacji?
rf_model = model.stages[-1]  # ostatni stage w Pipeline to nasz Random Forest
feature_names = ["location", "time_of_day", "year", "hour", "domestic"]
importances = rf_model.featureImportances

print("\n=== WAŻNOŚĆ CECH ===")
for name, imp in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
    bar = "█" * int(imp * 40)
    print(f"{name:<12} {imp:.4f}  {bar}")

# Przykładowe predykcje
print("\n=== PRZYKŁADOWE PREDYKCJE ===")
predictions.select("primary_type", "prediction", "label", "time_of_day", "Hour") \
    .show(10)

---
## Podsumowanie

| Krok | Co zrobiliśmy | Kluczowe API |
|------|---------------|--------------|
| 1 | Wczytanie, usunięcie duplikatów i null-i, parsowanie dat | `dropDuplicates()`, `dropna()`, `to_timestamp()` |
| 2 | Klasyfikacja pory dnia przez UDF | `@udf(StringType())`, `withColumn()` |
| 3 | Cache, broadcast join słownika kategorii, zapis do Parquet | `cache()`, `broadcast()`, `write.partitionBy().parquet()` |
| 4 | Agregacje, window functions, Spark SQL, `explain()` | `groupBy().agg()`, `Window`, `createOrReplaceTempView()` |
| 5 | Klasyfikacja wieloklasowa Random Forest | `Pipeline`, `StringIndexer`, `VectorAssembler`, `RandomForestClassifier` |

In [ ]:
# Zawsze zamknij sesję Sparka po zakończeniu pracy
spark.stop()
print("SparkSession zamknięta. Gotowe!")